In [15]:
import requests
from bs4 import BeautifulSoup
from typing import List, Optional, TypedDict
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ConfigDict
from langchain.chat_models import init_chat_model


load_dotenv()
llm = init_chat_model('gpt-4.1-mini')

In [16]:
class State(TypedDict):
    card_id: str
    card_name: Optional[str]
    raw_text: Optional[str]
    decision: Optional[str]      # 'direct' | 'selection'
    options_list: List[str]      # AI가 찾아낸 후보군 이름들
    summaries: str               # 사용자에게 보여줄 패키지별 요약 텍스트 (추가)
    selected_option: Optional[str] # 사용자가 최종 선택한 패키지명 (추가 권장)
    critic_feedback: Optional[dict]
    output_json: Optional[dict]

In [17]:
from typing import List, Optional, Union
from pydantic import BaseModel, Field, ConfigDict

# OpenAI Strict 모드를 위한 공통 설정
STRICT_CONF = ConfigDict(extra='forbid')

class OptionSummary(BaseModel):
    model_config = STRICT_CONF
    option_name: str
    brief_description: str # 예: "스타벅스 50% 할인 + 편의점 7% 할인"

class OptionsPreview(BaseModel):
    model_config = STRICT_CONF
    summaries: List[OptionSummary]

# 판단 노드(Decision Node) 전용 스키마
class FlowDecision(BaseModel):
    model_config = STRICT_CONF
    decision: str = Field(description="단일 카드면 'direct', 선택이 필요하면 'selection'")
    options: List[str] = Field(default=[], description="선택형일 경우 후보군 카드 이름 리스트")
    reason: str = Field(description="그렇게 판단한 이유")

# 1. 최하위 계층부터 정의 (거꾸로 올라갑니다)
class PerformanceTier(BaseModel):
    model_config = STRICT_CONF
    tier_min: int
    limit: int

class BenefitLimit(BaseModel):
    model_config = STRICT_CONF
    monthly: Optional[int] = None
    yearly: Optional[int] = None
    monthly_performance_tiers: Optional[List[PerformanceTier]] = None

class BenefitCondition(BaseModel):
    model_config = STRICT_CONF
    min_performance: int = 0
    payment_method: str = "ANY"
    is_overseas_only: Optional[bool] = None
    location_exclude: Optional[List[str]] = None
    per_transaction_cap: Optional[int] = None
    platform: Optional[str] = None
    domestic_only: Optional[bool] = None

class PerformanceImpact(BaseModel):
    model_config = STRICT_CONF
    counts_toward_performance: bool
    is_all_or_nothing_exclusion: bool = False
    comment: Optional[str] = None

class CardBenefit(BaseModel):
    model_config = STRICT_CONF
    benefit_id: str
    category: str
    merchant: Optional[List[str]] = None
    type: str
    value: float
    unit: str
    conditions: BenefitCondition
    limits: BenefitLimit
    performance_impact: PerformanceImpact

# 2. 중간 계층 (Logic 파트)
class PerformanceLogic(BaseModel):
    model_config = STRICT_CONF
    calculation_period: str
    global_performance_exclusion: List[str]
    domestic_only_performance: bool

class GracePeriodLogic(BaseModel):
    model_config = STRICT_CONF
    duration: str
    default_benefit_tier: str
    daily_life_limit: int
    air_dutyfree_limit: int
    transport_limit: int

# 3. 최상위 계층 (Root)
class CardDataSchema(BaseModel):
    model_config = STRICT_CONF
    card_id: str
    card_name: str
    issuer: str
    critical_warning: str
    performance_logic: PerformanceLogic
    benefits: List[CardBenefit]
    grace_period_logic: GracePeriodLogic

In [18]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate


# 2. 노드 함수 정의 (연산자 | 없이 invoke 직접 사용)

def node_ingest_card_data(state: State):
    card_id = state["card_id"]
    print(f"📡 [Node: Ingest] 카드 ID {card_id} 크롤링 시작...")
    
    url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36...",
    }
    
    # 실제 요청 수행
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"데이터 로드 실패: {response.status_code}")
        
    data = response.json()
    
    # 텍스트 정제 (BeautifulSoup 활용)
    benefit_segments = []
    for benefit in data.get("key_benefit", []):
        soup = BeautifulSoup(benefit.get("info", ""), "html.parser")
        benefit_segments.append(f"[{benefit.get('title')}]\n{soup.get_text(separator=' ').strip()}")
    
    return {
        "card_name": data.get("name"),
        "raw_text": "\n\n".join(benefit_segments)
    }

def node_decide_flow(state: State):
    print("🧐 [Node: Decide] 단일 카드인지 선택형인지 판단 중...")
    
    # 파이프(|) 연산자 대신 직접 invoke 호출
    structured_llm = llm.with_structured_output(FlowDecision)
    prompt = ChatPromptTemplate.from_template("""
    이 카드의 상세 텍스트를 보고 다음을 판단해:
    1. '기본 혜택(모든 사용자 공통)'과 '선택 혜택(여러 옵션 중 하나만 택일)'을 구분해.
    2. 만약 특정 이름(예: 라이프스타일 패키지) 아래 1~6번 혹은 A~F형 같은 '패키지'가 존재한다면, 그 패키지들이 진짜 'selection' 후보야.
    3. 단순히 카테고리(대중교통, 통신 등)가 나열된 것은 'selection'이 아니야.

    [텍스트]: {text}
    """)
    
    # 텍스트 채우기 및 호출
    formatted_input = prompt.format(text=state["raw_text"])
    response = structured_llm.invoke(formatted_input)
    
    return {
        "decision": response.decision,
        "options_list": response.options
    }

def node_extract_data(state: State):
    print(f"✨ [Node: Extract] '{state['card_name']}' 데이터 정밀 분석 중...")
    
    structured_llm = llm.with_structured_output(CardDataSchema)
    
    # 조금 더 친절한 가이드
    system_msg = """너는 신용카드 혜택 분석 전문가야. 
    제공된 텍스트에서 전월 실적 제외 항목, 구간별 한도, 유예 기간 로직을 정확히 찾아내서 JSON으로 변환해."""
    
    result = structured_llm.invoke(f"{system_msg}\n\n텍스트: {state['raw_text']}")
    
    # card_id 등 기본 정보가 누락되지 않도록 강제 주입 가능
    result_dict = result.model_dump()
    result_dict['card_id'] = state['card_id']
    
    return {"output_json": result_dict}

from langgraph.types import interrupt

def node_summarize_options(state: State):
    print("📝 [Node: Summarize] 선택지별 맛보기 정보 생성 중...")
    
    prompt = ChatPromptTemplate.from_template("""
    사용자에게 보여줄 선택지 요약을 만들어줘. 
    특히 '라이프스타일 패키지'처럼 번호가 매겨진 옵션이 있다면, 각 번호별로 혜택이 어떻게 다른지 대조해서 보여줘야 해.

    예: 패키지 1(스타벅스 50%, 오픈마켓 7%), 패키지 2(스타벅스 30%, 오픈마켓 7%)...

    [후보군]: {options_list}
    [텍스트]: {raw_text}
    """)
    
    summarizer = llm.with_structured_output(OptionsPreview)
    res = summarizer.invoke(prompt.format(
        options_list=state["options_list"],
        raw_text=state["raw_text"]
    ))
    
    # 요약 정보를 State에 저장 (State에 summaries 필드 추가 필요)
    return {"summaries": res.summaries}

from langgraph.types import interrupt

def node_wait_user_choice(state: State):
    # 1. 요약된 내용 출력 (summaries가 문자열이라면 그대로 출력)
    print("\n[AI 요약 리포트]")
    print(state["summaries"])
    
    # 2. 인터럽트 발생 (사용자 입력 대기)
    # CLI 환경이라면 여기서 멈추고 입력을 기다립니다.
    user_input = interrupt({
        "question": "분석할 패키지 번호(1~6)나 이름을 입력해주세요.",
        "options": state["options_list"]
    })
    
    # 3. 입력값 처리 (번호인지 이름인지 판단)
    selected = None
    
    # 입력이 숫자(1, 2, 3...)인 경우 처리
    if str(user_input).isdigit():
        idx = int(user_input) - 1
        if 0 <= idx < len(state["options_list"]):
            selected = state["options_list"][idx]
            print(f"✅ {idx+1}번({selected})이 선택되었습니다.")
    
    # 입력이 이름인 경우 처리
    if not selected:
        selected = user_input
        print(f"✅ '{selected}' 패키지가 선택되었습니다.")
    
    # 4. 다음 단계를 위해 State 업데이트
    # decision을 'direct'로 바꿔야 추출 노드로 넘어갑니다.
    return {
        "decision": "direct", 
        "selected_option": selected,
        "card_name": f"{state['card_name']} ({selected})"
    }

# 3. 라우팅 함수 (Conditional Edge용)
def router(state: State):
    if state["decision"] == "selection":
        return "선택필요"
    return "즉시추출"



In [19]:
# 그래프 조립 
graph = StateGraph(State)

# 1. 노드 등록
graph.add_node("수집", node_ingest_card_data)
graph.add_node("판단", node_decide_flow)
graph.add_node("요약", node_summarize_options) # 신규
graph.add_node("선택", node_wait_user_choice)  # 신규
graph.add_node("추출", node_extract_data)

# 2. 엣지 연결
graph.add_edge(START, "수집")
graph.add_edge("수집", "판단")

# 3. 조건부 분기 (판단 결과에 따라 요약으로 갈지 바로 추출로 갈지)
graph.add_conditional_edges(
    "판단",
    router,
    {
        "선택필요": "요약",
        "즉시추출": "추출"
    }
)

graph.add_edge("요약", "선택")
graph.add_edge("선택", "추출") # 선택이 완료되면 드디어 정밀 추출!
graph.add_edge("추출", END)

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

In [ ]:
# 1. 실행 설정을 정의합니다 (thread_id는 아무 문자열이나 상관없습니다)
config = {"configurable": {"thread_id": "card_analysis_session_01"}}

if __name__ == "__main__":
    initial_input = {"card_id": "51", "options_list": []}
    
    # 2. invoke 호출 시 config를 함께 넘겨줍니다
    # 이제 에러 없이 실행되며, '선택' 노드에서 멈출 것입니다.
    final_state = app.invoke(initial_input, config=config)
    
    # interrupt가 발생하면 여기서 실행이 멈춤.
    if final_state.get("output_json"):
        print("✅ 분석 완료!")

📡 [Node: Ingest] 카드 ID 51 크롤링 시작...
🧐 [Node: Decide] 단일 카드인지 선택형인지 판단 중...
📝 [Node: Summarize] 선택지별 맛보기 정보 생성 중...

[AI 요약 리포트]
[OptionSummary(option_name='라이프스타일 패키지 내 패키지 1', brief_description='스타벅스 50% 할인, 오픈마켓 7% 할인, 소셜커머스 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 2', brief_description='스타벅스 50% 할인, 소셜커머스 7% 할인, 오픈마켓 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 3', brief_description='스타벅스 50% 할인, 트렌디숍 7% 할인, 오픈마켓 1% 적립, 소셜커머스 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 4', brief_description='커피전문점 30% 할인, 오픈마켓 7% 할인, 소셜커머스 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 5', brief_description='커피전문점 30% 할인, 소셜커머스 7% 할인, 오픈마켓 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 6', brief_description='커피전문점 30% 할인, 트렌디숍 7% 할인, 오픈마켓 1% 적립, 소셜커머스 1% 적립')]


In [23]:
from langgraph.types import Command

# 사용자가 입력하고 싶은 번호 (예: "1")
user_input = "1" 

# 멈춘 지점부터 다시 시작 (동일한 config 사용)
final_result = app.invoke(Command(resume=user_input), config=config)

# 최종 결과물 출력
if final_result.get("output_json"):
    print("\n✅ 최종 분석이 완료되었습니다!")
    import json
    print(json.dumps(final_result["output_json"], indent=2, ensure_ascii=False))

Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]
Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]
Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]
Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]
Deserializing unregistered type __main__.OptionSummary from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'OptionSummary')]
Deserializing unregistered type __main__.OptionSummary 


[AI 요약 리포트]
[OptionSummary(option_name='라이프스타일 패키지 내 패키지 1', brief_description='스타벅스 50% 할인, 오픈마켓 7% 할인, 소셜커머스 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 2', brief_description='스타벅스 50% 할인, 소셜커머스 7% 할인, 오픈마켓 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 3', brief_description='스타벅스 50% 할인, 트렌디숍 7% 할인, 오픈마켓 1% 적립, 소셜커머스 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 4', brief_description='커피전문점 30% 할인, 오픈마켓 7% 할인, 소셜커머스 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 5', brief_description='커피전문점 30% 할인, 소셜커머스 7% 할인, 오픈마켓 1% 적립, 트렌디숍 1% 적립'), OptionSummary(option_name='라이프스타일 패키지 내 패키지 6', brief_description='커피전문점 30% 할인, 트렌디숍 7% 할인, 오픈마켓 1% 적립, 소셜커머스 1% 적립')]
✅ 1번(라이프스타일 패키지 내 다양한 패키지들 (패키지1~패키지6))이 선택되었습니다.
✨ [Node: Extract] '삼성카드 taptap O (라이프스타일 패키지 내 다양한 패키지들 (패키지1~패키지6))' 데이터 정밀 분석 중...

✅ 최종 분석이 완료되었습니다!
{
  "card_id": "51",
  "card_name": "Custom Card",
  "issuer": "Samsung Card",
  "critical_warning": "",
  "perform